# Online Retail — Unsupervised Learning Analysis
**Dataset:** UCI Online Retail Dataset  
**Goal:** Discover natural customer groups without predefined labels  
**Methods:** PCA → K-Means → Hierarchical Clustering → DBSCAN

### Key difference from classification notebook
- No temporal split needed — we are not predicting, we are discovering
- Uses the **full transaction history** per customer for richer features
- Segments emerge from the data naturally rather than being defined by us

---

## 0. Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

---
## 1. Load & Clean Data

In [ ]:
df = pd.read_excel('Online Retail.xlsx', dtype={'CustomerID': str})
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID', 'Description'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue']     = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
snapshot_date     = df['InvoiceDate'].max()

print(f'Clean dataset : {df.shape[0]:,} rows')
print(f'Date range    : {df["InvoiceDate"].min().date()} to {snapshot_date.date()}')
df.head()

---
## 2. Feature Engineering — Full History

Unlike the supervised notebooks, we use the **entire dataset** per customer.
14 features covering RFM, order patterns, behavioral, and product diversity.

In [ ]:
df['Is_Weekend'] = df['InvoiceDate'].dt.dayofweek >= 5
df['Is_Morning'] = df['InvoiceDate'].dt.hour < 12
df['Category']   = df['StockCode'].astype(str).str[:2]
df['Week']        = df['InvoiceDate'].dt.isocalendar().week.astype(int)

customer_df = df.groupby('CustomerID').agg(
    Recency          = ('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency        = ('InvoiceNo',    'nunique'),
    Monetary         = ('Revenue',      'sum'),
    Avg_Order_Value  = ('Revenue',      'mean'),
    Max_Order        = ('Revenue',      'max'),
    Std_Order        = ('Revenue',      'std'),
    Avg_Items        = ('Quantity',     'mean'),
    Unique_Products  = ('StockCode',    'nunique'),
    Unique_Categories= ('Category',     'nunique'),
    Top_Category_Pct = ('Category',     lambda x: x.value_counts().iloc[0]/len(x)),
    Pct_Weekend      = ('Is_Weekend',   'mean'),
    Pct_Morning      = ('Is_Morning',   'mean'),
    Active_Weeks     = ('Week',         'nunique'),
).reset_index()

tenure = df.groupby('CustomerID').agg(
    First_Purchase=('InvoiceDate','min'),
    Last_Purchase =('InvoiceDate','max')
).reset_index()
tenure['Tenure'] = (tenure['Last_Purchase'] - tenure['First_Purchase']).dt.days
customer_df = customer_df.merge(tenure[['CustomerID','Tenure']], on='CustomerID', how='left')
customer_df['Std_Order'] = customer_df['Std_Order'].fillna(0)
customer_df['Tenure']    = customer_df['Tenure'].fillna(0)

print(f'Customers : {customer_df.shape[0]:,}')
print(f'Features  : {customer_df.shape[1]-1}')
customer_df.describe().round(2)

---
## 3. Log Transform & Scale

All skewed features log-transformed, then StandardScaler applied.
Scaling is essential for PCA, K-Means, and DBSCAN.

In [ ]:
log_cols = ['Recency','Frequency','Monetary','Avg_Order_Value','Max_Order',
            'Std_Order','Avg_Items','Unique_Products','Unique_Categories',
            'Active_Weeks','Tenure']
for col in log_cols:
    customer_df[f'Log_{col}'] = np.log1p(customer_df[col])

feature_cols = [f'Log_{c}' for c in log_cols] + \
               ['Top_Category_Pct','Pct_Weekend','Pct_Morning']

X        = customer_df[feature_cols].copy()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'\nFeatures: {feature_cols}')

---
## 4. PCA — Dimensionality Reduction

PCA reduces 14 correlated features into uncorrelated components.
It tells us:
1. How many dimensions actually carry signal
2. Which original features drive the most variance

In [ ]:
pca_full   = PCA()
pca_full.fit(X_scaled)
explained  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

n_80 = np.argmax(cumulative >= 0.80) + 1
n_90 = np.argmax(cumulative >= 0.90) + 1

print(f'{n_80} components explain 80% of variance')
print(f'{n_90} components explain 90% of variance')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, len(explained)+1), explained,
            color='steelblue', edgecolor='white', alpha=0.8)
axes[0].plot(range(1, len(explained)+1), explained,
             'o-', color='coral', linewidth=2)
axes[0].set_title('Scree Plot — Variance per Component')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')

axes[1].plot(range(1, len(cumulative)+1), cumulative,
             'o-', color='steelblue', linewidth=2)
axes[1].axhline(0.80, color='red',    linestyle='--', linewidth=1, label='80% threshold')
axes[1].axhline(0.90, color='orange', linestyle='--', linewidth=1, label='90% threshold')
axes[1].axvline(n_80, color='red',    linestyle=':',  linewidth=1)
axes[1].fill_between(range(1, len(cumulative)+1), cumulative, alpha=0.1, color='steelblue')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.suptitle(f'PCA — {n_80} components explain 80% of variance', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature loadings — what drives PC1 and PC2?
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

loadings_df = pd.DataFrame(
    pca_2d.components_.T,
    columns=['PC1','PC2'],
    index=feature_cols
)

print('Top features driving PC1 (overall activity):')
print(loadings_df['PC1'].abs().sort_values(ascending=False).head(5))
print('\nTop features driving PC2 (order size):')
print(loadings_df['PC2'].abs().sort_values(ascending=False).head(5))

plt.figure(figsize=(7, 7))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('PCA Feature Loadings — PC1 & PC2')
plt.tight_layout()
plt.show()

# Use n_80 components for clustering
pca_opt = PCA(n_components=n_80)
X_pca   = pca_opt.fit_transform(X_scaled)
print(f'\nClustering will use {n_80} PCA components')

---
## 5. K-Means — Finding Optimal K

In [ ]:
inertias   = []
sil_scores = []
K_range    = range(2, 11)

for k in K_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, lbl))
    print(f'K={k:2d}  Inertia={km.inertia_:,.0f}  Silhouette={sil_scores[-1]:.4f}')

best_k = K_range[np.argmax(sil_scores)]
print(f'\nBest K by silhouette score: {best_k}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2)
axes[0].axvline(best_k, color='red', linestyle='--', linewidth=1,
                label=f'Best K={best_k}')
axes[0].set_title('Elbow Method — Inertia by K')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, 'o-', color='coral', linewidth=2)
axes[1].axvline(best_k, color='red', linestyle='--', linewidth=1,
                label=f'Best K={best_k}')
axes[1].set_title('Silhouette Score by K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.suptitle('K-Means — Finding Optimal Number of Clusters', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. K-Means — Final Clustering

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_pca)
customer_df['KMeans_Cluster'] = cluster_labels

print('Cluster sizes:')
print(customer_df['KMeans_Cluster'].value_counts().sort_index())

profile_cols = ['Recency','Frequency','Monetary','Unique_Products',
                'Tenure','Active_Weeks','Avg_Order_Value','Max_Order']
print('\nCluster profiles:')
print(customer_df.groupby('KMeans_Cluster')[profile_cols].mean().round(2))

# 2D PCA scatter coloured by cluster
colors_clust = ['steelblue','coral']
fig, ax = plt.subplots(figsize=(9, 7))
for c, color in enumerate(colors_clust):
    mask = cluster_labels == c
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
               c=color, alpha=0.4, s=10,
               label=f'Cluster {c}')
ax.set_title('K-Means Clusters in PCA Space')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles bar chart
profile = customer_df.groupby('KMeans_Cluster')[profile_cols].mean()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
colors = ['steelblue','coral']

for i, col in enumerate(profile_cols):
    vals = [profile.loc[c, col] for c in range(best_k)]
    bars = axes[i].bar(
        [f'Cluster {c}' for c in range(best_k)],
        vals, color=colors, edgecolor='white', width=0.5
    )
    axes[i].set_title(col)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + max(vals)*0.02,
                     f'{val:,.1f}', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('K-Means Cluster Profiles — What defines each group?', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette plot
sil_vals = silhouette_samples(X_pca, cluster_labels)
fig, ax  = plt.subplots(figsize=(9, 6))
y_lower  = 10

for i, color in enumerate(colors):
    sil_i   = np.sort(sil_vals[cluster_labels == i])
    size_i  = sil_i.shape[0]
    y_upper = y_lower + size_i
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_i,
                     facecolor=color, alpha=0.7, label=f'Cluster {i}')
    ax.text(-0.05, y_lower + 0.5*size_i, str(i))
    y_lower = y_upper + 10

avg_sil = silhouette_score(X_pca, cluster_labels)
ax.axvline(avg_sil, color='red', linestyle='--', linewidth=1.5,
           label=f'Avg silhouette = {avg_sil:.3f}')
ax.set_title('Silhouette Plot — Cluster Quality')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Hierarchical Clustering

In [ ]:
# Dendrogram on a sample of 200 customers
np.random.seed(42)
sample_idx     = np.random.choice(len(X_pca), 200, replace=False)
X_sample       = X_pca[sample_idx]
linkage_matrix = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(linkage_matrix, ax=ax, truncate_mode='level', p=5,
           leaf_rotation=90, leaf_font_size=8,
           color_threshold=linkage_matrix[-best_k+1, 2])
ax.axhline(linkage_matrix[-best_k+1, 2], color='red',
           linestyle='--', linewidth=1.5,
           label=f'Cut for {best_k} clusters')
ax.set_title(f'Dendrogram — Hierarchical Clustering (sample of 200 customers)')
ax.set_xlabel('Customer index')
ax.set_ylabel('Distance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full hierarchical clustering
hc = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hc_labels = hc.fit_predict(X_pca)
customer_df['HC_Cluster'] = hc_labels

print('Hierarchical cluster sizes:')
print(pd.Series(hc_labels).value_counts().sort_index())

# Compare K-Means vs Hierarchical agreement
agree    = (customer_df['KMeans_Cluster'] == customer_df['HC_Cluster']).sum()
disagree = len(customer_df) - agree
print(f'\nK-Means vs Hierarchical agreement: {agree:,}/{len(customer_df):,} ({agree/len(customer_df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie([agree, disagree],
       labels=[f'Agree ({agree:,})', f'Disagree ({disagree:,})'],
       colors=['steelblue','coral'], autopct='%1.1f%%', startangle=90)
ax.set_title('K-Means vs Hierarchical\nCluster Agreement')
plt.tight_layout()
plt.show()

---
## 8. DBSCAN — Density-Based Clustering & Outlier Detection

In [ ]:
db        = DBSCAN(eps=1.5, min_samples=5)
db_labels = db.fit_predict(X_pca)
customer_df['DBSCAN_Cluster'] = db_labels

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = (db_labels == -1).sum()

print(f'DBSCAN found: {n_clusters_db} clusters')
print(f'Outliers    : {n_noise} customers ({n_noise/len(db_labels)*100:.1f}%)')
print(f'\nCluster sizes:')
for lbl in sorted(set(db_labels)):
    count = (db_labels == lbl).sum()
    name  = 'Outliers' if lbl == -1 else f'Cluster {lbl}'
    print(f'  {name}: {count:,}')

# Plot DBSCAN in PCA space
palette = ['steelblue','coral','mediumseagreen','mediumpurple']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for lbl in sorted(set(db_labels)):
    mask  = db_labels == lbl
    color = 'black' if lbl == -1 else palette[lbl % len(palette)]
    label = 'Outliers' if lbl == -1 else f'Cluster {lbl}'
    alpha = 0.3 if lbl == -1 else 0.5
    size  = 5   if lbl == -1 else 8
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                    c=color, alpha=alpha, s=size, label=label)
axes[0].set_title('DBSCAN — Clusters & Outliers in PCA Space')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=9)

categories = [f'Cluster {i}' for i in range(n_clusters_db)] + ['Outliers']
counts_db  = [(db_labels == i).sum() for i in range(n_clusters_db)] + [n_noise]
colors_db  = palette[:n_clusters_db] + ['gray']
bars = axes[1].bar(categories, counts_db, color=colors_db, edgecolor='white')
axes[1].set_title(f'DBSCAN — Customer Distribution')
axes[1].set_ylabel('Number of customers')
for bar, count in zip(bars, counts_db):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 20,
                 f'{count}\n({count/len(db_labels)*100:.1f}%)',
                 ha='center', fontsize=9, fontweight='bold')

plt.suptitle('DBSCAN — Density-based Clustering & Outlier Detection', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Who are the outliers?
outlier_mask = db_labels == -1
normal_mask  = db_labels != -1

print('--- Outlier Profile vs Normal Customers ---')
compare_cols = ['Recency','Frequency','Monetary','Max_Order','Unique_Products']
comparison = pd.DataFrame({
    'Normal'  : customer_df[normal_mask][compare_cols].mean(),
    'Outliers': customer_df[outlier_mask][compare_cols].mean()
}).round(2)
print(comparison)

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(compare_cols))
width = 0.35
ax.bar(x - width/2, comparison['Normal'],   width, label='Normal',   color='steelblue', edgecolor='white')
ax.bar(x + width/2, comparison['Outliers'], width, label='Outliers', color='coral',     edgecolor='white')
ax.set_title('Outlier Profile vs Normal Customers')
ax.set_xticks(x)
ax.set_xticklabels(compare_cols)
ax.set_ylabel('Mean value')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Key Findings

### PCA results
- **5 components** explain 80% of variance — 14 features reduced to 5
- **PC1** is driven by Frequency, Active_Weeks, Monetary, Tenure — overall engagement
- **PC2** is driven by Avg_Order_Value, Avg_Items, Max_Order — order size/value
- Pct_Weekend and Pct_Morning carry almost no signal

### K-Means results
- **K=2** is the natural number of clusters (best silhouette score)
- The data has a clear binary split:

| Cluster | Size | Recency | Frequency | Monetary | Label |
|---|---|---|---|---|---|
| 0 | 2,131 | 39 days | 7.2 orders | £3,710 | Active customers |
| 1 | 2,207 | 143 days | 1.4 orders | £455 | Lapsed customers |

### Hierarchical clustering
- Dendrogram confirms the K=2 split with a clear large gap
- High agreement with K-Means validates the cluster structure is real

### DBSCAN results
- Found **2 main clusters** matching K-Means exactly
- Identified **82 outlier customers (1.9%)** — these are extreme high spenders
- Outlier avg Monetary: **£18,709** vs normal avg: **£1,733** — 10x higher
- These outliers are VIP customers worth treating as a separate segment entirely

### Business interpretation
The data naturally separates into two customer types:
- **Active (Cluster 0)** — recent, frequent, high-value. Focus: retention and upselling
- **Lapsed (Cluster 1)** — haven't bought recently, low frequency. Focus: win-back campaigns
- **VIP outliers** — extreme spenders. Focus: dedicated account management

### What this adds over supervised classification
- No predefined labels needed — segments emerged naturally
- Revealed VIP outliers that the equal-thirds classification missed
- PCA showed that shopping time patterns (weekend/morning) add no real signal
- The binary Active/Lapsed split is more natural than the Low/Medium/High forced segments